# How to create enforcable structured citations within RAG

In [1]:
from dotenv import load_dotenv

load_dotenv("../../.env")

import openai
from pydantic import BaseModel, Field
import instructor
import json

In [2]:
client = instructor.from_provider(
    "openai/gpt-5.4-nano",
    mode=instructor.Mode.RESPONSES_TOOLS #use responses rather than completions API
)

In [3]:
prompt = """You are a helpful assistant. 
Please answer the following question: 
What is your name?"""

In [4]:
# data classes for structure enforcement
class RAGContextUsed(BaseModel):
    id: str =Field(description="The ID of the item used to answer the questions")
    description: str = Field(description="Description of the item that belongs to the id and was used to answer the question.")

class RAGResponse(BaseModel):
    reasoning: str = Field(description="Reasoning behind the given answer")
    answer: str = Field(description="Answer to the question asked")
    citations: list[RAGContextUsed] = Field(description="Relevant list of items used to answer the question")


### Use RAG with structured output

In [5]:
from qdrant_client import QdrantClient

#### Embedding function

In [6]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

#### Retrieval function

In [7]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [8]:
def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [9]:
def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

In [10]:
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- If there are multiple products referenced list them out as a list.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt

In [11]:
def generate_answer(prompt):
    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning={"effort": "none"},
        response_model=RAGResponse
    )

    return response

In [12]:
def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "data_obj": answer,
        "answer": answer.answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "citations": answer.citations
    }
    return final_answer
    

In [13]:
output = rag_pipeline("I need a headphone")

In [14]:
output

{'data_obj': RAGResponse(reasoning='User asked generally for a headphone. Provided available products include multiple headphone types (wired iPhone earbuds, Bluetooth sleep headband, kids headphones, sports headband, Bluetooth earbuds). Need clarification on use case and device type; list options briefly from available products.', answer='What kind of headphone do you need? Here are the available options:\n- 2 Pack iPhone Wired Headphones with 3.5mm Jack, microphone & volume control (Apple MFi Certified)\n- Sleep Headphones Bluetooth Headband + sleep mask (wireless; noise/snprintf sleep aid; adjustable; washable; 10–12H)\n- Kids Wireless Headphones (Bluetooth 5.0 + 3.5mm jack), adjustable headband, built-in mic (up to 7H)\n- Bluetooth 5.3 Sports Headband with ENC mic (20+ hours playtime)\n- Bluetooth 5.2 Wireless Earbuds with charging case (Type-C; IP7 waterproof; 40H playtime)\n\nTell me: (1) wired or Bluetooth, (2) for sleep / sports / kids / general calls & music, and (3) your devi